# Nyaya — base-model shootout under one retriever

Six readers, same retriever, same 413 Eval-v1 questions, 768 new tokens (the 384-token
budget clipped 44/413 base answers). The question this answers: is Qwen2.5-3B the right
reader, and does any Apache-2.0 model of the same size beat it?

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, Input dataset
`jitendrajha98/nyaya-model-src` (the repository snapshot). Session 2 (Gemma, Llama) also
needs an `HF_TOKEN` secret on an account that accepted those models' terms.

Set `SESSION` below. Runtime ~5 GPU-hours per session; every run saves `predictions.jsonl`
so scoring can be redone on CPU forever.


In [ ]:
# --- setup -------------------------------------------------------------
import glob, os, shutil, subprocess, sys, time

SESSION = 1   # 1: base-768, nyaya-3b-v3-768, qwen3-4b     2: gemma-3-4b, llama-3.2-3b, phi-4-mini

# Kaggle "T4 x2" -> DataParallel splits tensors across devices. One GPU, always.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print("No HF_TOKEN secret (fine for session 1; gated models in session 2 need it):", exc)

# The repository comes in as a dataset snapshot, not a git clone: no GitHub dependency.
cands = glob.glob("/kaggle/input/nyaya-model-src/**/pyproject.toml", recursive=True)
assert cands, "Add Input -> Datasets -> jitendrajha98/nyaya-model-src"
SRC = os.path.dirname(cands[0])
WORK = "/kaggle/working/nyaya-model"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, "src")
print("source:", SRC)


def run(cmd):
    """Run a pipeline step and FAIL LOUDLY (a `!python` crash reads as success)."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed (exit {proc.returncode}): {' '.join(cmd)}")


run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-train.txt"])


In [ ]:
# --- build Eval-v1 + GPU preflight -------------------------------------
import torch

run([sys.executable, "scripts/25_build_eval_v1.py"])

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {name} (sm_{major}{minor}); torch {torch.__version__} supports {torch.cuda.get_arch_list()}")
if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{name} unsupported by this torch build -- switch the accelerator to GPU T4 x2")
print("preflight OK")


In [ ]:
# --- SMOKE FIRST: 8 questions, timed on the SECOND pass (first pass downloads the model)
COMMON = ["--dense", "--k", "8", "--max-new-tokens", "768", "--batch-size", "2"]
FULL_N, N_MODELS, BUDGET_S = 413, 3, 11 * 3600

run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", "none", "--limit", "8", "--label", "smoke", *COMMON])
t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", "none", "--limit", "8", "--label", "smoke", *COMMON])
per_q = (time.time() - t0) / 8
projected = per_q * FULL_N * N_MODELS
print(f"smoke: {per_q:.1f}s/question steady state -> projected {projected/3600:.1f} h for {N_MODELS} models (budget {BUDGET_S/3600:.0f} h)")
if projected > BUDGET_S:
    raise RuntimeError("projection exceeds the session budget -- emulated dtype or wrong GPU? fix before burning the session")


In [ ]:
# --- the runs ------------------------------------------------------------
RUNS = {
    1: [("Qwen/Qwen2.5-3B-Instruct", "base-768"),
        ("NyayaLabs98/nyaya-3b-v3", "nyaya-3b-v3-768"),
        ("Qwen/Qwen3-4B-Instruct-2507", "qwen3-4b")],
    2: [("google/gemma-3-4b-it", "gemma-3-4b"),
        ("meta-llama/Llama-3.2-3B-Instruct", "llama-3.2-3b"),
        ("microsoft/Phi-4-mini-instruct", "phi-4-mini")],
}
for model_id, label in RUNS[SESSION]:
    run([sys.executable, "scripts/26_eval_v1_run.py", "--model", model_id, "--adapter", "none",
         "--split", "all", "--label", label, *COMMON])


In [ ]:
# --- results: download these --------------------------------------------
import json, pathlib, shutil

out = pathlib.Path("/kaggle/working/nyaya-shootout-results")
out.mkdir(exist_ok=True)
results = json.load(open("reports/eval_v1_results.json", encoding="utf-8"))
print(f"{'run':<20}{'fact_recall':>12}{'citation':>10}{'all_facts':>10}")
for label, payload in results.items():
    m = payload["metrics"]
    print(f"{label:<20}{m['fact_recall']:>11.1%}{m['citation_accuracy']:>10.1%}{m['all_facts_accuracy']:>10.1%}")
shutil.copy("reports/eval_v1_results.json", out)
for pred in pathlib.Path("outputs/eval-v1").glob("*/predictions.jsonl"):
    if pred.parent.name != "smoke":
        shutil.copy(pred, out / f"{pred.parent.name}_predictions.jsonl")
shutil.make_archive("/kaggle/working/nyaya-shootout-results", "zip", out)
print("collected:", sorted(p.name for p in out.iterdir()))
